In [2]:
# Katakana Gairaigo Puzzle — corrected (no f-string interpolation issues)
from IPython.display import HTML, display
from textwrap import dedent
import uuid, json, random

uid = "kata_puzzle_" + uuid.uuid4().hex[:8]

WORDS = [
    {"en":"computer", "romaji":"konpyūta",   "jp":"コンピュータ"},
    {"en":"television","romaji":"terebi",    "jp":"テレビ"},
    {"en":"ice cream", "romaji":"aisukurīmu","jp":"アイスクリーム"},
    {"en":"coffee",    "romaji":"kōhī",      "jp":"コーヒー"},
    {"en":"taxi",      "romaji":"takushī",   "jp":"タクシー"},
    {"en":"restaurant","romaji":"resutoran", "jp":"レストラン"},
    {"en":"America",   "romaji":"Amerika",   "jp":"アメリカ"},
    {"en":"banana",    "romaji":"banana",    "jp":"バナナ"},
    {"en":"piano",     "romaji":"piano",     "jp":"ピアノ"},
    {"en":"camera",    "romaji":"kamera",    "jp":"カメラ"},
]

PIECES = [
    "ア","イ","ウ","エ","オ",
    "カ","キ","ク","ケ","コ",
    "サ","シ","ス","セ","ソ",
    "タ","チ","ツ","テ","ト",
    "ナ","ニ","ヌ","ネ","ノ",
    "ハ","ヒ","フ","ヘ","ホ",
    "マ","ミ","ム","メ","モ",
    "ヤ","ユ","ヨ",
    "ラ","リ","ル","レ","ロ",
    "ワ","ヲ","ン",
    "ガ","ギ","グ","ゲ","ゴ",
    "ザ","ジ","ズ","ゼ","ゾ",
    "ダ","ヂ","ヅ","デ","ド",
    "バ","ビ","ブ","ベ","ボ",
    "パ","ピ","プ","ペ","ポ",
    "ァ","ィ","ゥ","ェ","ォ","ャ","ュ","ョ","ッ","ー",
    "ヴ"
]

root_id = "gp-root-" + uid

html = dedent("""
<div id="gp-root" style="font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial;">
  <style>
    #gp-root h2 { margin: 8px 0 4px; }
    #gp-root .row { display:flex; gap:10px; align-items:center; flex-wrap:wrap; }
    #gp-root .panel { border:1px solid #e2e2e2; border-radius:12px; padding:10px; background:#fff; }
    #gp-root .prompt { display:flex; flex-direction:column; gap:4px; }
    #gp-root .prompt .en { font-size:1.15rem; font-weight:600; }
    #gp-root .prompt .romaji { color:#555; }
    #gp-root .target { min-height:52px; display:flex; gap:6px; align-items:center; flex-wrap:wrap; border:1px dashed #bbb; border-radius:10px; padding:8px; background:#fafafa; }
    #gp-root .target .char { font-size:1.6rem; padding:4px 8px; border-radius:8px; background:#f0f3ff; border:1px solid #b7c5ff; }
    #gp-root .controls button, #gp-root .pool button { cursor:pointer; border-radius:10px; border:1px solid #ccc; background:#fff; padding:8px 10px; }
    #gp-root .controls button:hover, #gp-root .pool button:hover { background:#f6f6f6; }
    #gp-root .pool { display:grid; grid-template-columns: repeat(auto-fill, minmax(46px, 1fr)); gap:6px; }
    #gp-root .pool .piece { font-size:1.2rem; padding:8px 0; }
    #gp-root .status { font-size:.95rem; color:#444; min-height:1.2em; }
    #gp-root .ok { color:#0a8a0a; }
    #gp-root .bad { color:#b00020; }
    #gp-root .hint { color:#666; font-size:.92rem; }
    #gp-root .ratebox { margin-left:10px; display:flex; align-items:center; gap:6px; }
    #gp-root .voiceinfo { font-size:.9rem; color:#444; }
  </style>

  <h2>🧩 Katakana Gairaigo Puzzle</h2>
  <div class="panel">
    <div class="row">
      <div class="prompt">
        <div class="en" id="gp-board">—</div>
        <div class="romaji hint" id="gp-romaji">—</div>
      </div>
      <div class="row" style="margin-left:auto;">
        <span class="voiceinfo" id="gp-voiceinfo">Voice: (detecting Google Japanese…)</span>
        <div class="ratebox">
          <label for="gp-rate"><b>Speed:</b></label>
          <input type="range" id="gp-rate" min="0.7" max="1.5" step="0.05" value="1.00">
          <span id="gp-rateval">1.00×</span>
        </div>
      </div>
    </div>

    <div id="gp-target" class="target" aria-live="polite"></div>

    <div class="row controls" style="margin-top:10px;">
      <button id="gp-check">✔ Check</button>
      <button id="gp-play">▶ Play</button>
      <button id="gp-back">⌫ Backspace</button>
      <button id="gp-clear">🧹 Clear</button>
      <button id="gp-shuffle">🔀 Shuffle</button>
      <button id="gp-new">🎲 New word</button>
    </div>
    <div id="gp-status" class="status"></div>
  </div>

  <div class="panel" style="margin-top:12px;">
    <div class="hint" style="margin-bottom:6px;">Select the katakana pieces to match the loanword.</div>
    <div id="gp-pool" class="pool"></div>
  </div>

  <script>
  (function(){
    const WORDS = """ + json.dumps(WORDS, ensure_ascii=False) + """;
    const ALL   = """ + json.dumps(PIECES, ensure_ascii=False) + """;

    const boardEl   = document.getElementById("gp-board");
    const romajiEl  = document.getElementById("gp-romaji");
    const targetEl  = document.getElementById("gp-target");
    const poolEl    = document.getElementById("gp-pool");
    const statusEl  = document.getElementById("gp-status");
    const newBtn    = document.getElementById("gp-new");
    const checkBtn  = document.getElementById("gp-check");
    const clearBtn  = document.getElementById("gp-clear");
    const backBtn   = document.getElementById("gp-back");
    const shuffleBtn= document.getElementById("gp-shuffle");
    const playBtn   = document.getElementById("gp-play");
    const voiceInfo = document.getElementById("gp-voiceinfo");
    const rate      = document.getElementById("gp-rate");
    const rateVal   = document.getElementById("gp-rateval");

    let currentVoice = null;
    let current = null;
    let answer = [];

    function updateRateLabel(){
      const r = parseFloat(rate.value) || 1.0;
      rateVal.textContent = r.toFixed(2) + "×";
    }
    rate.addEventListener("input", updateRateLabel);
    updateRateLabel();

    function pickGoogleJa(list){
      const lower = s => (s||"").toLowerCase();
      let v = list.find(v => lower(v.name).includes("google") && v.lang && v.lang.toLowerCase().startsWith("ja"));
      if (v) return v;
      v = list.find(v => v.lang && v.lang.toLowerCase().startsWith("ja"));
      return v || list[0] || null;
    }
    function describe(v){ return v ? "Voice: " + (v.name||"?") + " — " + (v.lang||"?") : "Voice: (none)"; }

    function tryLoadVoicesOnce(){
      const list = speechSynthesis.getVoices() || [];
      if (list.length){
        currentVoice = pickGoogleJa(list);
        voiceInfo.textContent = describe(currentVoice);
        return true;
      }
      return false;
    }
    function ensureVoice(){
      if (tryLoadVoicesOnce()) return;
      try { const warm = new SpeechSynthesisUtterance(" "); warm.volume=0; speechSynthesis.speak(warm); } catch(e){}
      let tries = 0;
      const t = setInterval(()=>{
        tries++;
        if (tryLoadVoicesOnce() || tries>12) clearInterval(t);
      },200);
    }
    if (speechSynthesis.addEventListener){
      speechSynthesis.addEventListener("voiceschanged", tryLoadVoicesOnce);
    } else {
      speechSynthesis.onvoiceschanged = tryLoadVoicesOnce;
    }
    ensureVoice();

    function currentRate(){
      const r = parseFloat(rate.value);
      return Math.max(0.7, Math.min(1.5, isNaN(r)?1.0:r));
    }
    function speakJa(text){
      const t = (text||"").trim();
      if (!t) return;
      const u = new SpeechSynthesisUtterance(t);
      u.lang = "ja-JP";
      if (currentVoice) u.voice = currentVoice;
      u.rate = currentRate();
      speechSynthesis.cancel();
      speechSynthesis.speak(u);
    }

    function choice(arr){ return arr[Math.floor(Math.random()*arr.length)]; }
    function shuffle(a){ for(let i=a.length-1;i>0;i--){ const j=Math.floor(Math.random()*(i+1)); [a[i],a[j]]=[a[j],a[i]]; } return a; }

    function renderTarget(){ targetEl.innerHTML = answer.map(ch=>"<span class='char'>"+ch+"</span>").join(""); }
    function setStatus(msg,good=false,bad=false){
      statusEl.textContent = msg||"";
      statusEl.classList.remove("ok","bad");
      if (good) statusEl.classList.add("ok");
      if (bad) statusEl.classList.add("bad");
    }

    function fillPool(){
      const pieces = shuffle(ALL.slice());
      poolEl.innerHTML = pieces.map(ch=>"<button class='piece' data-ch='"+ch+"'>"+ch+"</button>").join("");
    }

    function newWord(){
      current = choice(WORDS);
      answer = [];
      boardEl.textContent = current.en;
      romajiEl.textContent = "(" + current.romaji + ")";
      renderTarget();
      setStatus("Build the katakana word.");
      fillPool();
    }

    poolEl.addEventListener("click", e=>{
      const b = e.target.closest("button.piece"); if(!b) return;
      answer.push(b.getAttribute("data-ch"));
      renderTarget();
      setStatus("");
    });

    backBtn.addEventListener("click", ()=>{ answer.pop(); renderTarget(); });
    clearBtn.addEventListener("click", ()=>{ answer=[]; renderTarget(); setStatus("Cleared."); });
    shuffleBtn.addEventListener("click", ()=> fillPool());
    newBtn.addEventListener("click", ()=> newWord());

    checkBtn.addEventListener("click", ()=>{
      const guess = answer.join("");
      if (!current) return;
      if (guess === current.jp){
        setStatus("Correct! " + guess, true, false);
      } else {
        setStatus("Not yet. You made: " + guess + " (target: " + current.jp + ")", false, true);
      }
    });

    playBtn.addEventListener("click", ()=>{
      const guess = answer.join("");
      speakJa(guess || (current ? current.jp : ""));
    });

    newWord();
  })();
  </script>
</div>
""")

display(HTML(html))
